In [0]:
!find . -type d -name "__pycache__" -exec rm -r {} +

In [0]:
%pip install pyspark pandas psycopg2-binary

In [0]:
%restart_python

In [0]:
import sys
from pathlib import Path

# Ajusta la ruta a tu estructura de carpetas:
PROJECT_ROOT = Path.cwd().parent  
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Directorio raíz del proyecto agregado:", PROJECT_ROOT)
from m01_data_ingestion import ingest as ingest_data

# Ejecuta la ingesta
df = ingest_data()
print("Primeras filas del DataFrame crudo:")
display(df.head())

#Prueba Modulos step

### Step01

In [0]:
# Prueba para step01_import: ParquetPartitionLoader

from m05_feature_engineering.step01_import import ParquetPartitionLoader

# Instancia el loader (usa la ruta por defecto configurada en el módulo)
loader = ParquetPartitionLoader()

# Carga los datasets particionados
X_train, X_test, X_backtest, y_train, y_test, y_backtest = loader.load()

# Muestra el shape de cada partición
print(f"X_train:    {X_train.shape}, y_train:    {y_train.shape}")
print(f"X_test:     {X_test.shape},  y_test:     {y_test.shape}")
print(f"X_backtest: {X_backtest.shape}, y_backtest: {y_backtest.shape}")

# Opcional: muestra las primeras filas para validar visualmente
print("\nPrimeras filas de X_train:")
print(X_train.head())
print("\nPrimeras filas de y_train:")
print(y_train.head())


### step02

In [0]:
import pandas as pd
import numpy as np

# 1. Creamos tres dataframes de juguete (pueden ser idénticos)
data = {
    "edad":   [25, np.nan, 30, 29, np.nan],
    "peso":   [70, 68, np.nan, 75, 72],
    "altura": [1.65, 1.7, 1.68, np.nan, 1.72]
}

train_df    = pd.DataFrame(data)
test_df     = pd.DataFrame(data)
backtest_df = pd.DataFrame(data)

print("Datos originales (train):")
print(train_df)

# 2. Prueba de imputación usando tu step02_imputation
from m05_feature_engineering.step02_imputation import SimpleImputerAdapter

imputer = SimpleImputerAdapter()

# Ajustamos/imputamos sobre train
train_imputed = imputer.fit_transform(train_df.copy())
# Imputamos sobre test y backtest
test_imputed = imputer.transform(test_df.copy())
backtest_imputed = imputer.transform(backtest_df.copy())

# 3. Mostramos resultados
print("\nTrain imputado:\n", train_imputed)
print("\nTest imputado:\n", test_imputed)
print("\nBacktest imputado:\n", backtest_imputed)


### step03

In [0]:
import pandas as pd
from m05_feature_engineering.step03_outliers import IQRHandler

# DataFrame base con outliers
data = {
    "edad":   [25, 26, 27, 28, 100],   # 100 es un outlier
    "peso":   [70, 68, 69, 71, 150],   # 150 es un outlier
    "altura": [1.65, 1.70, 1.68, 1.66, 2.5]  # 2.5 es un outlier
}

train_df    = pd.DataFrame(data)
test_df     = pd.DataFrame(data)
backtest_df = pd.DataFrame(data)

print("=== Datos originales ===")
print(train_df)

# Instancia y ajusta sobre train
handler = IQRHandler(factor=1.5)
train_capped = handler.fit_transform(train_df.copy())

# Aplica transform sobre test y backtest
test_capped = handler.transform(test_df.copy())
backtest_capped = handler.transform(backtest_df.copy())

print("\n=== Train tras capping ===")
print(train_capped)

print("\n=== Test tras capping ===")
print(test_capped)

print("\n=== Backtest tras capping ===")
print(backtest_capped)


### step04

In [0]:
import pandas as pd
from m05_feature_engineering.step04_transformation import StandardScaleTransformer

# 1. Creamos dataframes de juguete
data = {
    "edad":   [20, 30, 40, 60, 100],
    "ingresos": [1000, 1200, 2000, 5000, 15000]
}
train_df    = pd.DataFrame(data)
test_df     = pd.DataFrame(data)
backtest_df = pd.DataFrame(data)

print("=== Datos originales (train) ===")
print(train_df)

# 2. Instanciamos y aplicamos el transformador
transformer = StandardScaleTransformer()

# Ajustamos sobre train
train_trans = transformer.fit_transform(train_df.copy())

# Transformamos test y backtest usando los parámetros aprendidos en train
test_trans     = transformer.transform(test_df.copy())
backtest_trans = transformer.transform(backtest_df.copy())

print("\n=== Train transformado ===")
print(train_trans)

print("\n=== Test transformado ===")
print(test_trans)

print("\n=== Backtest transformado ===")
print(backtest_trans)


### step05 -> step08

In [0]:
import pandas as pd
from m05_feature_engineering.step05_encoding  import OneHotEncoderAdapter
from m05_feature_engineering.step06_feature_gen import PolynomialFeatureGenerator
from m05_feature_engineering.step07_metrics    import JsonMetricsExporter
from m05_feature_engineering.step08_drift      import PSIDriftDetector

# ------------------------------------------------------------------
# 1) Creamos DataFrames de juguete (train / test / backtest)
data = {
    "sexo": ["M", "F", "F", "M", "F"],
    "ciudad": ["A", "B", "A", "C", "B"],
    "edad": [25, 30, 35, 28, 40],
    "ingresos": [1000, 1500, 1200, 1100, 3000],
    "target": [0, 1, 0, 0, 1]
}
train_df    = pd.DataFrame(data)
test_df     = pd.DataFrame(data)
backtest_df = pd.DataFrame(data)

# Separamos features y target
X_train, y_train = train_df.drop("target", axis=1), train_df["target"]
X_test,  y_test  = test_df.drop("target", axis=1),  test_df["target"]
X_back,  y_back  = backtest_df.drop("target", axis=1), backtest_df["target"]

# ------------------------------------------------------------------
# 2) One-Hot Encoding
encoder = OneHotEncoderAdapter()
X_train_enc = encoder.fit_transform(X_train.copy())
X_test_enc  = encoder.transform(X_test.copy())
X_back_enc  = encoder.transform(X_back.copy())

print("Columns after OHE:", X_train_enc.columns.tolist())

# ------------------------------------------------------------------
# 3) Generación de polinomios grado 2
poly_gen = PolynomialFeatureGenerator(degree=2)
X_train_poly = poly_gen.fit_transform(X_train_enc.copy())
X_test_poly  = poly_gen.transform(X_test_enc.copy())
X_back_poly  = poly_gen.transform(X_back_enc.copy())

print("\nShape after PolynomialFeatures:")
print("  Train:", X_train_poly.shape, "Test:", X_test_poly.shape)

# ------------------------------------------------------------------
# 4) Exportamos métricas del train transformado
metrics = JsonMetricsExporter()
metrics.export(pd.concat([X_train_poly, y_train], axis=1), "metrics/feature_metrics.json")
print("\n✅ Métricas guardadas en metrics/feature_metrics.json")

# ------------------------------------------------------------------
# 5) Detectamos drift entre train y test
drift_detector = PSIDriftDetector(bins=10)
psi_scores = drift_detector.compute(X_train_poly, X_test_poly)
print("\nPSI por columna:\n", psi_scores)


### pipeline completo

In [0]:
from m05_feature_engineering.pipeline_engineering import run_pipeline
run_pipeline()
from pathlib import Path
import pandas as pd
# 1) Calcula la carpeta data/processed relativa al cwd (src/app)
base = Path.cwd().parent / "data" / "processed"

# 2) Carga cada Parquet
df_train   = pd.read_parquet(base / "X_train_processed.parquet")
df_test    = pd.read_parquet(base / "X_test_processed.parquet")
df_back    = pd.read_parquet(base / "X_backtest_processed.parquet")

# 3) Muestra un vistazo
print("Train procesado:\n", df_train.head(), "\n")
print("Test procesado:\n", df_test.head(), "\n")
print("Backtest procesado:\n", df_back.head())


# Feature selection

In [0]:
from m06__feature_selection.step01_import import ParquetPartitionLoader2

loader = ParquetPartitionLoader2()          # usa la ruta por defecto corregida
X_train, X_test, X_back, y_train, y_test, y_back = loader.load()

print("Shapes:", X_train.shape, X_test.shape, X_back.shape)


In [0]:
# %% [markdown]
# ## Test de step02_filtering.filter_partitions

# %%
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# 1) Asegura que src/app esté en sys.path
APP_DIR = Path().resolve().parent  # si el notebook está en src/app/test
if str(APP_DIR) not in sys.path:
    sys.path.append(str(APP_DIR))

# %%
# 2) Importa la función
from m06__feature_selection.step02_filtering import filter_partitions

# %%
# 3) Datos de ejemplo
rng = np.random.default_rng(0)
X = pd.DataFrame({
    "const": 1,
    "a": rng.normal(size=50),
    "b": None,  # se llenará como alta correlación con 'a'
    "c": rng.normal(size=50),
})
X["b"] = X["a"] * 0.95 + rng.normal(scale=0.1, size=50)
# Divide en train/test/back
X_train = X.iloc[:30]
X_test = X.iloc[30:40]
X_back = X.iloc[40:]
y_train = pd.Series(rng.integers(0, 2, size=30))

# %%
# 4) Aplica filter_partitions
X_tr_f, X_te_f, X_ba_f, pipe = filter_partitions(X_train, X_test, X_back, y_train)

# %%
# 5) Resultados
print("Columnas finales:", X_tr_f.columns.tolist())
print("Shapes:", X_tr_f.shape, X_te_f.shape, X_ba_f.shape)
X_tr_f.head()


In [0]:
# %% [markdown]
# # Prueba de `frame_partitions` con estimadores de clasificación y regresión

# %%
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# 1) Asegura que src/app esté en sys.path
APP_DIR = Path().resolve()
if str(APP_DIR) not in sys.path:
    sys.path.append(str(APP_DIR))

# 2) Importa
from m06__feature_selection.step03_frame import frame_partitions

# %% [markdown]
# ## 2.1 Caso clasificación (default LogisticRegression)

# %%
# Datos sintéticos de clasificación
rng = np.random.default_rng(0)
n = 100
Xc = pd.DataFrame({
    "a": rng.normal(size=n),
    "b": rng.normal(size=n),
    "c": rng.normal(size=n),
    "d": rng.normal(size=n) + np.arange(n)*0.1,
    "e": rng.integers(0,2,size=n),
})
yc = (Xc["d"]*2 + Xc["e"]*5 + rng.normal(scale=1,size=n) > 10).astype(int)

# Particiones
split1 = int(0.6 * n)
split2 = int(0.8 * n)
Xc_tr, Xc_te, Xc_ba = (Xc.iloc[:split1], Xc.iloc[split1:split2], Xc.iloc[split2:])
yc_tr = yc[:split1]

# Aplica selector por defecto
Xc_tr_f, Xc_te_f, Xc_ba_f, sel_c = frame_partitions(
    Xc_tr, Xc_te, Xc_ba, yc_tr
)

print("Clasificación → columnas seleccionadas:", sel_c.selected_cols)
print("Shapes:", Xc_tr_f.shape, Xc_te_f.shape, Xc_ba_f.shape)

# %% [markdown]
# ## 2.2 Caso regresión (LinearRegression)

# %%
from sklearn.linear_model import LinearRegression

# Datos sintéticos de regresión
Xr = pd.DataFrame({
    "x1": rng.normal(size=n),
    "x2": rng.normal(size=n),
    "x3": rng.normal(size=n),
    "x4": rng.normal(size=n) * 5 + np.arange(n)*0.1,
    "x5": rng.normal(size=n)*2,
})
yr = Xr["x4"] * 2 + Xr["x5"] * -3 + rng.normal(scale=5, size=n)

# Particiones
Xr_tr, Xr_te, Xr_ba = (Xr.iloc[:split1], Xr.iloc[split1:split2], Xr.iloc[split2:])
yr_tr = yr[:split1]

# Aplica selector con regresor
Xr_tr_f, Xr_te_f, Xr_ba_f, sel_r = frame_partitions(
    Xr_tr, Xr_te, Xr_ba, yr_tr,
    estimator=LinearRegression(),
    forward_k=4,
    final_k=2,
)

print("Regresión → columnas seleccionadas:", sel_r.selected_cols)
print("Shapes:", Xr_tr_f.shape, Xr_te_f.shape, Xr_ba_f.shape)

# 3) Inspección rápida
display(Xr_tr_f.head())


In [0]:
%pip install abess

In [0]:
%restart_python

In [0]:
# %% [markdown]
# # Test robusto de abess_partitions (regresión temporal)

# %%
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Asegura que src/app esté en sys.path
APP_DIR = Path().resolve()
if str(APP_DIR) not in sys.path:
    sys.path.append(str(APP_DIR))

# Importa la función
from m06__feature_selection.step04_abess import abess_partitions

# Simula un dataframe semanal con target continua
rng = np.random.default_rng(42)
n_weeks = 104
df = pd.DataFrame({
    "semana": np.arange(n_weeks),
    "clima": rng.normal(size=n_weeks),
    "publicidad": rng.integers(0, 2, size=n_weeks),
    "eventos": rng.poisson(0.2, size=n_weeks),
    "vacaciones": (np.arange(n_weeks) % 12 == 0).astype(int),
})
df["nuevos_pacientes"] = (
    30 + 2*df["clima"] + 10*df["publicidad"] - 5*df["vacaciones"]
    + rng.normal(scale=3, size=n_weeks)
)

# Separa features y target
X = df.drop(columns=["semana", "nuevos_pacientes"])
y = df["nuevos_pacientes"]

# Particiona 60% train / 20% test / 20% backtest
idx1 = int(0.6 * n_weeks)
idx2 = int(0.8 * n_weeks)
X_train = X.iloc[:idx1].reset_index(drop=True)
X_test  = X.iloc[idx1:idx2].reset_index(drop=True)
X_back  = X.iloc[idx2:].reset_index(drop=True)
y_train = y.iloc[:idx1].reset_index(drop=True)

# Ejecuta la selección (mode="regression")
X_tr_f, X_te_f, X_ba_f, selector = abess_partitions(
    X_train, X_test, X_back, y_train, mode="regression"
)

print("Seleccionadas:", selector.selected_cols)
print("Shapes:", X_tr_f.shape, X_te_f.shape, X_ba_f.shape)
display(X_tr_f.head())
